# Foundational SAT Skill Gap Prediction Using Kernel k-NN and Label Propagation

**Synthetic data only.** No real student data.

## Flow (response-based)
1. Synthetic MCQ **item bank** (3 items × 72 skills)
2. **100 students** attempt items for **~40 skills** each → **response matrix R**
3. Aggregate: **S = normalize(R · Q)** → skill mastery %
4. Untested skills = no items attempted (~32 missing per student)
5. **Gaussian kernel** k-NN on shared known skills + **label propagation** on skill graph W → predict gaps
6. Priority ranking for untested foundational skills

In [ ]:
import numpy as np
import pandas as pd
from lib import (
    KERNEL_SIGMA,
    PROPAGATION_ALPHA,
    K_NEIGHBORS,
    build_predicted_missing_skills_df,
    build_skill_affinity_matrix,
    gaussian_kernel,
    kernel_similarity_observed,
    run_pipeline,
    analyze_student,
)
from export_frontend import export_dashboard
from items import ITEMS, write_item_bank

## 1. Item bank & response matrix R

Each item maps to one skill. Matrix **R** has shape (students × items); entry is 1 if correct, 0 if wrong, NaN if not attempted.

In [ ]:
result = run_pipeline(seed=42)
print(f"Items: {len(result.item_ids)}, Students: {len(result.student_ids)}")
print(f"Response matrix R shape: {result.response_matrix.shape}")
print(f"Skill-item matrix Q shape: {result.skill_item_matrix.shape}")
result.responses_df.head()

## 2. Aggregate to skill matrix S

$$S_{i,j} = 100 \times \frac{\sum_k R_{i,k} Q_{k,j}}{\sum_k Q_{k,j} \cdot \mathbb{1}[R_{i,k} \text{ attempted}]}$$

Implemented as group-by: % correct items per (student, skill).

In [ ]:
S = result.scores
M = result.mask
print(f"Skill matrix S: {S.shape}")
print(f"Tested skills per student: {M.sum(axis=1).mean():.1f} avg")
print(f"Untested fraction: {(~M).mean():.1%}")

## 3. Sample student work (auditable)

Every displayed mastery score equals % correct from these item attempts.

In [ ]:
target_idx = result.default_student_idx
sid = result.student_ids[target_idx]
work = result.responses_df[result.responses_df.student_id == sid]
print(f"Student {sid}: {len(work)} items across {M[target_idx].sum()} skills")
work[["skill_name", "item_id", "chosen", "correct_choice", "is_correct"]].head(12)

## 4. Gaussian kernel on shared skills

On shared known skills Ω:
$$\|u - v\|^2 = (u - v)^\top (u - v)$$
$$K(u,v) = \exp\left(-\frac{\|u - v\|^2}{2\sigma^2}\right), \quad \sigma = \text{25}$$

Skill affinity matrix **W** (row-normalized) encodes prerequisites, same category, and adjacent difficulty levels for label propagation.

In [ ]:
analysis = analyze_student(result, target_idx)
peer_idx = 1
shared = M[target_idx] & M[peer_idx]
u, v = S[target_idx, shared], S[peer_idx, shared]
k_val, sq_dist = kernel_similarity_observed(S[target_idx], S[peer_idx], shared, KERNEL_SIGMA)
print(f"Squared distance ||u-v||^2: {sq_dist:.2f}")
print(f"Gaussian kernel K(u,v): {k_val:.4f}  (sigma={KERNEL_SIGMA})")
print(f"Skill affinity W shape: {result.skill_affinity.shape}")
print(f"k={K_NEIGHBORS}, alpha={PROPAGATION_ALPHA}")

## 5. k-NN + label propagation → predictions & priority ranking

For each missing skill j:
$$\text{neighbor\_pred}_j = \frac{\sum_i K_i \cdot \text{peer\_score}_{i,j}}{\sum_i K_i}$$
$$\text{related\_pred}_j = \frac{\sum_{j'} W_{j,j'} \cdot \text{known\_score}_{j'}}{\sum_{j'} W_{j,j'}}$$
$$\text{final}_j = \alpha \cdot \text{neighbor\_pred} + (1-\alpha) \cdot \text{related\_pred}$$

In [ ]:
recs = pd.DataFrame(analysis["recommendations"])
missing_df = build_predicted_missing_skills_df(
    analysis["predictions"], M, target_idx, result.skills, analysis["breakdown"], sid
)
print(analysis["summary"]["interpretation"])
print(f"\nMissing skills predicted: {len(missing_df)}")
recs.head(10)[["rank", "skill_name", "neighbor_pred", "related_pred", "predicted_mastery", "priority_score"]]

In [ ]:
from lib import save_figures
save_figures(result, target_idx, analysis)
export_dashboard()
print("Figures + dashboard.json exported")

## Assignment checklist

- [x] **Student-skill matrix** S ∈ ℝ^{100×72} (incomplete)
- [x] **Student vectors** with ~40 known coordinates
- [x] **Distance / norm:** ‖u−v‖² via dot product on shared skills
- [x] **Gaussian kernel** K(u,v) = exp(−‖u−v‖² / 2σ²)
- [x] **Weighted k-NN** prediction using kernel weights
- [x] **Label propagation** from skill affinity graph W
- [x] **Missing-entry prediction** blending collaborative + structural signals
- [x] **Priority ranking** for foundational untested skills

**Real problem:** Prioritize untested foundational skills when full coverage is impossible.

**Verifiable artifacts:** Item bank, response CSV, skill scores, `predicted_missing_skills.csv`, figures, dashboard JSON.

*Optional future extension: PCA/SVD — not used here.*